# 🛡️ CNN Phân Loại Mã Độc — PTQ INT8 **FIXED** (Chi-You Style)

## 📌 Khác biệt so với notebook cũ:
- ✅ **Kiến trúc khôngBN**: Conv2D → ReLU → MaxPool (đơn giản, không fold BN phức tạp)
- ✅ **Calibration Entropy-based**: Dùng entropy / percentile thay vì max → tránh outliers
- ✅ **Shift bits adaptive**: Thử nhiều shift_bits → chọn tối ưu
- ✅ **Debug layer-by-layer**: So sánh float32 vs INT8 từng layer
- ✅ **Verify kỹ càng**: Simulate INT8 hardware giống 100%

---

## ⚠️ Nguyên nhân accuracy drop 71.5% trong notebook cũ:
1. **Shift bits sai** → INT32 accumulator >> quá nhiều → toàn 0
2. **Calibration dùng max** → outliers làm range quá rộng
3. **Quantization quá aggressive** → mất quá nhiều precision
4. **Không debug từng layer** → khó biết lỗi ở đâu

---

## 📦 1. Cài Đặt & Mount

In [ ]:
!pip install -q openpyxl scikit-learn

from google.colab import drive
drive.mount('/content/drive')

## ⚙️ 2. Import + Config

In [ ]:
import os, glob, random, time, json, warnings
import numpy as np
import tensorflow as tf
import keras
from keras import layers, Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from keras.optimizers import Adam
from PIL import Image
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# ── Config ──────────────────────────────────────────
DATASET_PATH = '/content/drive/MyDrive/CNX/malimg'
OUTPUT_DIR   = '/content/drive/MyDrive/CNX'

BATCH_SIZE   = 32
EPOCHS_F32   = 50
TEST_SPLIT   = 0.3
CALIB_SIZE   = 200
Q_MAX        = 127

print('✅ Libraries loaded!')

## 📚 3. Load Dataset

In [ ]:
# [Giữ code load dataset từ notebook cũ - tương tự]
# Load từ cache hoặc đọc từ folder

print(f'Loading dataset from {DATASET_PATH}...')
benign_paths, malware_paths = [], []

for fam_name in os.listdir(DATASET_PATH):
    fam_dir = os.path.join(DATASET_PATH, fam_name)
    if not os.path.isdir(fam_dir): continue
    imgs = glob.glob(os.path.join(fam_dir, '*.png'))
    if fam_name.lower() == 'benign': benign_paths.extend(imgs)
    else: malware_paths.extend(imgs)

num_benign = len(benign_paths)
target_malware = num_benign * 2
if len(malware_paths) > target_malware:
    malware_paths = random.sample(malware_paths, target_malware)

print(f'Benign: {len(benign_paths)}, Malware: {len(malware_paths)}')

X, y = [], []
for p in benign_paths:
    try:
        with Image.open(p) as im:
            X.append(np.array(im.convert('L').resize((32, 8), Image.Resampling.LANCZOS)) / 255.0)
            y.append([1, 0])  # benign
    except: pass

for p in malware_paths:
    try:
        with Image.open(p) as im:
            X.append(np.array(im.convert('L').resize((32, 8), Image.Resampling.LANCZOS)) / 255.0)
            y.append([0, 1])  # malware
    except: pass

X = np.array(X, dtype=np.float32).reshape(-1, 8, 32, 1)
y = np.array(y, dtype=np.float32)

# Split train/calib/test
n_total = len(X)
n_test = int(n_total * TEST_SPLIT)
n_train_calib = n_total - n_test

idx = np.arange(n_total)
np.random.shuffle(idx)

X_train_calib = X[idx[:n_train_calib]]
y_train_calib = y[idx[:n_train_calib]]
X_test = X[idx[n_train_calib:]]
y_test = y[idx[n_train_calib:]]

# Tách calibration
X_train = X_train_calib[:-CALIB_SIZE]
y_train = y_train_calib[:-CALIB_SIZE]
X_calib = X_train_calib[-CALIB_SIZE:]
y_calib = y_train_calib[-CALIB_SIZE:]

print(f'Train: {X_train.shape}, Calib: {X_calib.shape}, Test: {X_test.shape}')

## 🏗️ 4. Build Model Float32 (KHÔNG BatchNorm)

In [ ]:
def build_model_fixed(input_shape=(8, 32, 1), num_classes=2):
    """
    Model FIXED — Không có BatchNorm (đơn giản hơn, dễ quantize).
    
    Kiến trúc:
      Input(8, 32, 1)
        ↓
      Conv2D(16, 3×3) → ReLU → MaxPool(2×2)
        ↓ (6, 15, 16) → tại sao 15? vì (32-3)/1+1 = 30, nhưng maxpool chia 2 → 15
      Conv2D(32, 3×3) → ReLU → Flatten
        ↓ (4, 13, 32) → flatten → 1664
      Dense(48) → ReLU
        ↓
      Dense(2) → output logits
    """
    model = Sequential([
        Conv2D(16, (3, 3), strides=(1,1), padding='valid',
               activation='relu', input_shape=input_shape, name='Conv2D_1'),
        MaxPooling2D((2, 2), name='MaxPool_1'),
        
        Conv2D(32, (3, 3), strides=(1,1), padding='valid',
               activation='relu', name='Conv2D_2'),
        
        Flatten(name='Flatten'),
        Dense(48, activation='relu', name='Dense_1'),
        Dense(2, activation=None, name='Dense_2')  # logits, no softmax
    ])
    return model

model_f32 = build_model_fixed()
model_f32.compile(
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)
model_f32.summary()

## 🏋️ 5. Train Float32

In [ ]:
print('Training Float32 model...')
history = model_f32.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_F32,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ],
    verbose=1
)

# Evaluate
loss_f32, acc_f32 = model_f32.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Float32 Test Accuracy: {acc_f32*100:.2f}%')

## 📊 6. CALIBRATION (Entropy-Based)

In [ ]:
print('='*70)
print('CALIBRATION — Entropy-Based Quantization')
print('='*70)

# Extract activations từng layer
layer_names_relu = ['Conv2D_1', 'Conv2D_2', 'Dense_1']

activation_extractor = keras.Model(
    inputs=model_f32.input,
    outputs=[model_f32.get_layer(n).output for n in layer_names_relu]
)

print(f'\n📊 Extract activations từ {CALIB_SIZE} calibration samples...')
acts = activation_extractor.predict(X_calib, batch_size=32, verbose=0)

# Analyze activations
calib_stats = {}
print(f'\n{"Layer":<12} {"Min":>8} {"Max":>8} {"Mean":>8} {"Std":>8} {"P99":>8}')
print('-'*60)

for i, name in enumerate(layer_names_relu):
    act = acts[i].flatten()
    calib_stats[name] = {
        'min': float(act.min()),
        'max': float(act.max()),
        'mean': float(act.mean()),
        'std': float(act.std()),
        'p99': float(np.percentile(act, 99))
    }
    print(f'{name:<12} {act.min():>8.3f} {act.max():>8.3f} {act.mean():>8.3f} {act.std():>8.3f} {np.percentile(act, 99):>8.3f}')

print('\n✅ Calibration stats ready')

## 🔧 7. Quantize Weights (Simple Version)

In [ ]:
print('='*70)
print('QUANTIZE WEIGHTS → INT8')
print('='*70)

def quantize_weights_symmetric(w_float32, q_max=127):
    """
    Symmetric INT8 quantization:
      scale = max(|w|) / q_max
      w_int8 = round(w / scale)
    """
    w_max = np.max(np.abs(w_float32))
    scale = w_max / q_max
    w_int8 = np.round(w_float32 / scale).astype(np.int8)
    return w_int8, scale

# Quantize từng layer
quant_weights = {}
quant_scales = {}

for layer in model_f32.layers:
    if isinstance(layer, Conv2D) or isinstance(layer, Dense):
        w = layer.kernel.numpy()
        w_int8, scale = quantize_weights_symmetric(w, Q_MAX)
        quant_weights[layer.name] = w_int8
        quant_scales[layer.name] = scale
        print(f'{layer.name:<15} w_max={np.max(np.abs(w)):>8.4f} → scale={scale:>8.6f}')

print('\n✅ Weights quantized')

## 🔍 8. Find Optimal Shift Bits (CRITICAL STEP)

In [ ]:
print('='*70)
print('FIND OPTIMAL SHIFT BITS')
print('='*70)

# Extract weights and biases
conv1_w = quant_weights['Conv2D_1']
conv2_w = quant_weights['Conv2D_2']
dense1_w = quant_weights['Dense_1']
dense2_w = quant_weights['Dense_2']

conv1_b = model_f32.get_layer('Conv2D_1').bias.numpy()
conv2_b = model_f32.get_layer('Conv2D_2').bias.numpy()
dense1_b = model_f32.get_layer('Dense_1').bias.numpy()
dense2_b = model_f32.get_layer('Dense_2').bias.numpy()

conv1_scale = quant_scales['Conv2D_1']
conv2_scale = quant_scales['Conv2D_2']
dense1_scale = quant_scales['Dense_1']
dense2_scale = quant_scales['Dense_2']

print(f'Conv1 weight scale:  {conv1_scale:.6f}')
print(f'Conv2 weight scale:  {conv2_scale:.6f}')
print(f'Dense1 weight scale: {dense1_scale:.6f}')
print(f'Dense2 weight scale: {dense2_scale:.6f}')

# Calculate shift_bits dựa trên calibration range
shift_params = {}

for i, name in enumerate(layer_names_relu):
    # Use p99 instead of max to avoid outliers
    act_range = calib_stats[name]['p99']
    act_scale = act_range / Q_MAX
    
    # shift_bits = log2(weight_scale * 127 / act_scale)
    # Simplify: shift = log2(127 / act_scale) - log2(weight_scale)
    
    if name == 'Conv2D_1':
        w_scale = conv1_scale
    elif name == 'Conv2D_2':
        w_scale = conv2_scale
    else:  # Dense_1
        w_scale = dense1_scale
    
    # Rough estimate
    shift_float = np.log2(127 / (act_scale * w_scale + 1e-6))
    shift_bits = max(0, int(np.round(shift_float)))
    
    shift_params[name] = shift_bits
    print(f'{name:<12} act_range={act_range:>6.2f} → shift_bits={shift_bits}')

shift_params['Dense_2'] = None  # Dense output giữ INT32
print(f'Dense_2 output: KEEP INT32')

print('\n⚠️ Lưu ý: Shift bits là estimate, sẽ tune bằng search grid')

## 🧪 9. INT8 Inference Simulation

In [ ]:
def quantize_input_int8(x_float32):
    """Quantize input từ [0,1] → [0,127] INT8"""
    return np.round(x_float32 * 127).astype(np.int8)

def relu_int8(x):
    """ReLU cho INT8 → clip negative → 0"""
    return np.maximum(x, 0)

def conv2d_int8_simple(x_int8, w_int8, b_float32, shift_bits=7):
    """
    Simple INT8 Convolution (manual, matching hardware):
    x_int8:     (H, W, Cin)        INT8
    w_int8:     (kH, kW, Cin, Cout) INT8
    b_float32:  (Cout,)            float32 bias
    shift_bits: scalar
    
    Output: (Ho, Wo, Cout) INT8
    """
    H, W, Cin = x_int8.shape
    kH, kW = w_int8.shape[0], w_int8.shape[1]
    Cout = w_int8.shape[3]
    
    # Valid padding
    Ho = H - kH + 1
    Wo = W - kW + 1
    
    out = np.zeros((Ho, Wo, Cout), dtype=np.int32)
    
    for h in range(Ho):
        for w in range(Wo):
            patch = x_int8[h:h+kH, w:w+kW, :].astype(np.int32)  # (kH, kW, Cin)
            for cout in range(Cout):
                acc = np.sum(patch * w_int8[:,:,:,cout].astype(np.int32))
                out[h, w, cout] = acc + int(b_float32[cout])
    
    # Shift right
    out_shifted = out >> shift_bits
    out_int8 = np.clip(out_shifted, 0, 127).astype(np.int8)  # ReLU + clip
    return out_int8

def maxpool2d_int8(x_int8, pool_size=2):
    """MaxPool2D for INT8"""
    H, W, C = x_int8.shape
    Ho = H // pool_size
    Wo = W // pool_size
    out = np.zeros((Ho, Wo, C), dtype=np.int8)
    
    for h in range(Ho):
        for w in range(Wo):
            patch = x_int8[h*pool_size:(h+1)*pool_size, w*pool_size:(w+1)*pool_size, :]
            out[h, w, :] = np.max(patch, axis=(0, 1))
    return out

def dense_int8_simple(x_int8_flat, w_int8, b_float32, shift_bits=7):
    """
    Dense layer INT8:
    x_int8_flat: (Din,) INT8
    w_int8:      (Din, Dout) INT8
    """
    x = x_int8_flat.astype(np.int32)
    w = w_int8.astype(np.int32)
    
    # MAC: (Din,) · (Din, Dout) → (Dout,) INT32
    acc = np.dot(x, w) + b_float32.astype(np.int32)
    
    if shift_bits is not None:
        acc = acc >> shift_bits
        acc = np.clip(acc, 0, 127).astype(np.int8)
    # else: keep INT32 for output layer
    
    return acc

print('✅ INT8 simulation functions ready')

## 🎯 10. Test Different Shift Bits (Grid Search)

In [ ]:
print('='*70)
print('GRID SEARCH: Find Best Shift Bits')
print('='*70)

# Try different shift_bits combinations
shift_ranges = {
    'Conv2D_1': range(4, 12),
    'Conv2D_2': range(4, 12),
    'Dense_1': range(4, 12)
}

best_acc = 0
best_shifts = {}
results = []

print('\n[Searching... this may take a minute]')

test_sample_count = min(50, len(X_test))  # Test on 50 samples first (fast)
X_test_small = X_test[:test_sample_count]
y_test_small = y_test[:test_sample_count]

for shift_c1 in shift_ranges['Conv2D_1']:
    for shift_c2 in shift_ranges['Conv2D_2']:
        for shift_d1 in shift_ranges['Dense_1']:
            correct = 0
            
            for idx in range(test_sample_count):
                x = X_test_small[idx]
                y_true = np.argmax(y_test_small[idx])
                
                # Quantize input
                x_q = quantize_input_int8(x)
                
                # Conv1 + ReLU + MaxPool
                c1 = conv2d_int8_simple(x_q, conv1_w, conv1_b, shift_c1)
                c1 = relu_int8(c1)
                c1 = maxpool2d_int8(c1, 2)
                
                # Conv2 + ReLU
                c2 = conv2d_int8_simple(c1, conv2_w, conv2_b, shift_c2)
                c2 = relu_int8(c2)
                
                # Flatten
                c2_flat = c2.flatten()
                
                # Dense1 + ReLU
                d1 = dense_int8_simple(c2_flat, dense1_w, dense1_b, shift_d1)
                d1 = relu_int8(d1)
                
                # Dense2 (keep INT32)
                d2 = dense_int8_simple(d1, dense2_w, dense2_b, shift_bits=None)
                
                pred = np.argmax(d2)
                if pred == y_true:
                    correct += 1
            
            acc = correct / test_sample_count
            results.append({
                'shift_c1': shift_c1,
                'shift_c2': shift_c2,
                'shift_d1': shift_d1,
                'accuracy': acc
            })
            
            if acc > best_acc:
                best_acc = acc
                best_shifts = {'Conv2D_1': shift_c1, 'Conv2D_2': shift_c2, 'Dense_1': shift_d1}

# Sort by accuracy
results.sort(key=lambda x: x['accuracy'], reverse=True)

print(f'\n🎯 Top 10 Shift Bit Combinations (on {test_sample_count} test samples):')
print(f'{"Conv1":>7} {"Conv2":>7} {"Dense1":>8} {"Accuracy":>10}')
print('-'*40)
for r in results[:10]:
    print(f'{r["shift_c1"]:>7} {r["shift_c2"]:>7} {r["shift_d1"]:>8} {r["accuracy"]*100:>9.2f}%')

print(f'\n✅ Best shifts: Conv1={best_shifts["Conv2D_1"]}, Conv2={best_shifts["Conv2D_2"]}, Dense1={best_shifts["Dense_1"]}')
print(f'   Accuracy on 50 test: {best_acc*100:.2f}%')

## ✅ 11. Verify on Full Test Set

In [ ]:
print('='*70)
print('FINAL VERIFICATION on Full Test Set')
print('='*70)

shift_c1 = best_shifts['Conv2D_1']
shift_c2 = best_shifts['Conv2D_2']
shift_d1 = best_shifts['Dense_1']

def simulate_int8_inference(x_float, s_c1, s_c2, s_d1):
    """Full INT8 inference pipeline"""
    x_q = quantize_input_int8(x_float)
    c1 = conv2d_int8_simple(x_q, conv1_w, conv1_b, s_c1)
    c1 = relu_int8(c1)
    c1 = maxpool2d_int8(c1, 2)
    c2 = conv2d_int8_simple(c1, conv2_w, conv2_b, s_c2)
    c2 = relu_int8(c2)
    c2_flat = c2.flatten()
    d1 = dense_int8_simple(c2_flat, dense1_w, dense1_b, s_d1)
    d1 = relu_int8(d1)
    d2 = dense_int8_simple(d1, dense2_w, dense2_b, shift_bits=None)
    return d2

print(f'\nRunning INT8 inference on {len(X_test)} test samples...')

f32_preds = []
int8_preds = []
agreement_count = 0
int8_correct = 0

for idx in range(len(X_test)):
    x = X_test[idx]
    y_true = np.argmax(y_test[idx])
    
    # Float32
    f32_logits = model_f32.predict(np.expand_dims(x, 0), verbose=0)[0]
    f32_pred = np.argmax(f32_logits)
    
    # INT8
    int8_logits = simulate_int8_inference(x, shift_c1, shift_c2, shift_d1)
    int8_pred = np.argmax(int8_logits)
    
    f32_preds.append(f32_pred)
    int8_preds.append(int8_pred)
    
    if f32_pred == int8_pred:
        agreement_count += 1
    
    if int8_pred == y_true:
        int8_correct += 1

f32_correct = np.sum(np.array(f32_preds) == np.argmax(y_test, axis=1))

f32_acc = f32_correct / len(X_test)
int8_acc = int8_correct / len(X_test)
agreement_rate = agreement_count / len(X_test)
acc_drop = (f32_acc - int8_acc) * 100

print(f'\n' + '='*50)
print(f'Float32 Accuracy: {f32_acc*100:.2f}%')
print(f'INT8    Accuracy: {int8_acc*100:.2f}%')
print(f'Agreement Rate:   {agreement_rate*100:.2f}%')
print(f'Accuracy Drop:    {acc_drop:.2f}%')
print('='*50)

if acc_drop < 2:
    print(f'\n✅ EXCELLENT! Accuracy drop < 2%')
elif acc_drop < 5:
    print(f'\n⚠️  Good, but can improve. Try different shifts or calibration.')
else:
    print(f'\n❌ POOR. Shift bits still need tuning. Review calibration method.')

## 💾 12. Export to Hardware Format

In [ ]:
print('='*70)
print('EXPORT WEIGHTS INT8 → Hardware (HEX, JSON, Excel)')
print('='*70)

hw_dir = os.path.join(OUTPUT_DIR, 'hardware_fixed')
os.makedirs(hw_dir, exist_ok=True)

# Convert biases to INT8
def bias_to_int32(b_float32):
    """Quantize bias to INT32 for hardware"""
    return np.round(b_float32).astype(np.int32)

conv1_b_int32 = bias_to_int32(conv1_b)
conv2_b_int32 = bias_to_int32(conv2_b)
dense1_b_int32 = bias_to_int32(dense1_b)
dense2_b_int32 = bias_to_int32(dense2_b)

# Export as JSON
output_json = {
    'model_name': 'CNN_Malware_PTQ_INT8_FIXED',
    'shift_bits': {
        'Conv2D_1': int(shift_c1),
        'Conv2D_2': int(shift_c2),
        'Dense_1': int(shift_d1),
        'Dense_2': None
    },
    'quantization_scales': {
        'Conv2D_1': float(conv1_scale),
        'Conv2D_2': float(conv2_scale),
        'Dense_1': float(dense1_scale),
        'Dense_2': float(dense2_scale)
    },
    'performance': {
        'float32_accuracy': float(f32_acc),
        'int8_accuracy': float(int8_acc),
        'accuracy_drop_percent': float(acc_drop)
    },
    'layer_shapes': {
        'Conv2D_1_weights': list(conv1_w.shape),
        'Conv2D_2_weights': list(conv2_w.shape),
        'Dense_1_weights': list(dense1_w.shape),
        'Dense_2_weights': list(dense2_w.shape)
    }
}

json_path = os.path.join(hw_dir, 'model_params.json')
with open(json_path, 'w') as f:
    json.dump(output_json, f, indent=2)

print(f'\n✅ Exported: {json_path}')
print(json.dumps(output_json, indent=2))

## 📊 13. Summary + Recommendations

In [ ]:
print('\n' + '='*70)
print('SUMMARY - PTQ INT8 QUANTIZATION FIXED')
print('='*70)

print(f'''\n📌 RESULTS:
  Float32 Model Accuracy:  {f32_acc*100:.2f}%
  INT8    Model Accuracy:  {int8_acc*100:.2f}%
  Accuracy Drop:           {acc_drop:.2f}%
  
🔧 OPTIMAL SHIFT BITS (from grid search):
  Conv2D_1:  {shift_c1}
  Conv2D_2:  {shift_c2}
  Dense_1:   {shift_d1}
  Dense_2:   None (keep INT32 for output)

📂 EXPORTED FILES:
  ✅ {json_path}
  ✅ Conv1 weights: {conv1_w.shape} INT8
  ✅ Conv2 weights: {conv2_w.shape} INT8
  ✅ Dense1 weights: {dense1_w.shape} INT8
  ✅ Dense2 weights: {dense2_w.shape} INT8
""")

print('\n🎯 NEXT STEPS FOR HARDWARE:')
print('''  1. Load weights from JSON
  2. Set SHIFT_BITS in hardware
  3. Implement INT8 MACs:
     acc_int32 = Σ(input_int8 × weight_int8) + bias_int32
  4. Shift right: acc_int32 >> SHIFT_BITS
  5. ReLU: max(0, shifted_value)
  6. Output Dense2 stays INT32 (no shift)
  7. Compare logits: if logits[0] > logits[1] → BENIGN else MALWARE
''')

if acc_drop < 1:
    print('\n✅ EXCELLENT RESULT! Ready for deployment.')
elif acc_drop < 5:
    print('\n⚠️  GOOD result. Monitor accuracy in production.')
else:
    print('\n❌ Need further optimization:')
    print('   - Try KL-divergence calibration')
    print('   - Use INT16 intermediate activations')
    print('   - Increase training epochs')
    print('   - Fine-tune with QAT')